# 5. Modelos del lenguaje (estadísticos)

![](https://lena-voita.github.io/resources/lectures/lang_models/examples/morphosyntax-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

## Objetivo

- Implementar modelos del lenguaje estadísticos
- Aplicaciones

## Modelos del lenguaje

> "Un modelo del lenguaje es un modelo estadístico que asigna probabilidades a cadenas dentro de un lenguaje" - Parafraseando a mi compadre [Jurafsky, 2026](https://web.stanford.edu/~jurafsky/slp3/3.pdf)

$$ \mu = (\Sigma, A, \Pi)$$

Donde:
- $\mu$ es el modelo del lenguaje
- $\Sigma$ es el vocabulario
- $A$ es el tensor que guarda las probabilidades
- $\Pi$ guarda las probabilidades iniciales

- Este modelo busca estimar la probabilidad de una secuencia de tokens
- Pueden ser palabras, caracteres o tokens
- Se pueden considerar varios escenarios para la creación de estos modelos
    - Si podemos estimar la probabilidad de una unidad lingüística (palabras, tokens, oraciones, etc), podemos usarlar de formas insospechadas

### Probabilidad de una oración

- El objetivo es estimar las probabilidades de unidades lingüísticas que reflejen el comportamiento del lenguaje natural
- Esto es, por ejemplo, las oraciones que tengan mayor probabilidad de ocurrir

####  Muchas oraciones probablemente no ocurran en nuestro corpus. ¿Cómo lidiamos con eso❓

#### I saw a cat in a mat

<img src="https://lena-voita.github.io/resources/lectures/lang_models/general/i_saw_a_cat_prob.gif">

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

Sean $y_1, y_2, \dots, y_n$ tokens y $P(y_1, y_2, \dots, y_n)$ la probabilidad de verlos en ese orden. Si aplicamos la regla de la cadena obtenemos:

$$
P(y_1, y_2, \dots, y_n)=P(y_1)\cdot P(y_2|y_1) \cdot P(y_3|y_1, y_2)\cdot\dots\cdot P(y_n|y_1, \dots, y_{n-1})=
        \prod \limits_{t=1}^n P(y_t|y_{<t}).
$$

Con esto modelamos la probabilidad de que un conjunto de tokens ocurran como una **probabilidad condicional** $P(y_n|y_1, \dots, y_{n-1})$. Podemos estimar esta probabilidad de multiples formas:

- N-gramas
- Modelos neuronales

### Sobre bigramas y N-gramas

- Para bigramas tenemos la propiedad de Markov
- Para $n > 2$ las palabras dependen de mas elementos
    - Trigramas
    - 4-gramas
- En general para un modelo de n-gramas se toman en cuenta $n-1$ elementos

![](https://lena-voita.github.io/resources/lectures/lang_models/ngram/example_cut_3gram-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

## Programando nuestros modelos del lenguaje

Utilizaremos un [corpus](https://www.nltk.org/book/ch02.html) en inglés disponible en NLTK

In [ ]:
import nltk

nltk.download("gutenberg")
nltk.download("abc")
nltk.download("genesis")
nltk.download("inaugural")
nltk.download("state_union")
nltk.download("webtext")
nltk.download("punkt_tab")

In [ ]:
import numpy as np

In [ ]:
from nltk.corpus import abc, genesis, gutenberg, inaugural, state_union, webtext

# Exploración del corpus
total_sents = 0
corpora = []
plaintext_corpora = {
    "abc": abc,
    "Gutenberg": gutenberg,
    "Genesis": genesis,
    "Inaugural": inaugural,
    "State Union": state_union,
    "Web": webtext,
}
for title, corpus in plaintext_corpora.items():
    corpus_sents = corpus.sents()
    corpus_len = len(corpus_sents)
    total_sents += corpus_len
    print(f"{title} sents={corpus_len}")
    corpora.append(corpus_sents)
print(f"Total={total_sents}")

In [ ]:
def preprocess_sent(sent: list[str]) -> list[str]:
    """Función de preprocesamiento

    Agrega tokens de inicio y fin a la oración y normaliza todo a minusculas

    Params
    ------
    sent: list[str]
        Lista de palabras que componen la oración

    Return
    ------
    Las oración preprocesada
    """
    result = [word.lower() for word in sent]
    # Al final de la oración
    result.append("</s>")
    # Al inicio de la oración
    result.insert(0, "<s>")
    return result

In [ ]:
print(preprocess_sent(corpora[0][0]))

In [ ]:
for i, sent in enumerate(corpora[0][:10]):
    print(i, " ".join(sent))

In [ ]:
for i, sent in enumerate(corpora[0][:10]):
    print(i, " ".join(preprocess_sent(sent)))

In [ ]:
from nltk import ngrams

for ngram in list(ngrams(preprocess_sent(corpora[0][10]), 3))[:10]:
    print(ngram)

### Construyendo el modelo del lenguaje

![](https://imgs.xkcd.com/comics/predictive_models_2x.png)

In [ ]:
from collections import Counter, defaultdict  # ❤️‍🔥❤️‍🔥❤️‍🔥

test = defaultdict(Counter)

In [ ]:
test[("hola", "que")]["hace"] += 1

In [ ]:
test

In [16]:
type NgramModel = defaultdict[tuple[str, str], Counter[str]]

In [17]:
def build_trigram_model(data: list[list[list[str]]]) -> NgramModel:
    # Initialize model as a nested dict with default behavior
    model: NgramModel = defaultdict(Counter)
    for corpus in data:
        for sentence in corpus:
            for w1, w2, w3 in ngrams(preprocess_sent(sentence), 3):
                model[(w1, w2)][w3] += 1
    return model

In [ ]:
%%time
trigram_model = build_trigram_model(corpora)

In [ ]:
trigram_model["<s>", "the"]

### Calculo de probabilidades con conteos de trigramas

In [ ]:
for i, key in enumerate(trigram_model):
    print(key)
    if i == 5:
        break

In [18]:
def calculate_model_probs(model: NgramModel) -> NgramModel:
    model_probs = defaultdict(Counter)
    # Por cada prefijo del modelo
    for prefix in model:
        # Todas las veces que ocurre prefix seguido de cualquier palabra
        total = float(sum(model[prefix].values()))
        # Por cada palabra w que haya ocurrido con prefix
        for w in model[prefix]:
            # Obtenemos la probabilidad
            model_probs[prefix][w] = model[prefix][w] / total
    return model_probs

## Smoothing 🥤

![](https://lena-voita.github.io/resources/lectures/lang_models/ngram/prob_cat_on_a_mat-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

¿Qué pasaría se no sucede la secuencia de tokens del númerador/denominador? Para evitar este problema se utiliza una técnica llamada *smoothing* que redistribuye la función de masa de probabilidad.

#### ¿Cómo agregamos smoothing❓

La forma más sencilla es pretender que vimos al menos una vez todos los n-gramas. Esto es sumar 1 a todas las cuentas. Algo más general sería agregar una cantidad $\delta$:

$$
P(mat| cat\ on\ a) = \frac{\delta + N(cat\ on\ a\ mat)}{\delta \cdot |V| + N(cat\ on\ a)}
$$

#### Ejercicio 🤹🏽: Implementa el smoothing de Laplace agregando $\delta$ a todas las cuentas de los n-gramas

- *Hint*: Obten todos los tokens, despues el vocabulario y el tamaño del vocabulario

In [ ]:
# Calcula el vocabulario acá
VOCABULARY_SIZE = 0

In [ ]:
def calculate_smooth_probs(model: NgramModel, vocab_size: int, delta: float = 1.0) -> NgramModel:
    model_probs: NgramModel = defaultdict(Counter)
    # Tu código bonito acá 🥸
    return model_probs

In [ ]:
trigram_probs = calculate_model_probs(trigram_model)

In [ ]:
trigram_smooth = calculate_smooth_probs(trigram_model, VOCABULARY_SIZE)

In [ ]:
sorted(dict(trigram_probs["<s>", "the"]).items(), key=lambda x: -1 * x[1])

In [ ]:
sorted(dict(trigram_smooth["<s>", "the"]).items(), key=lambda x: -1 * x[1])

### Aplicaciones

- Generación de texto
- Completado de texto
- Speech To Text (STT)

![](https://lena-voita.github.io/resources/lectures/lang_models/examples/suggest-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

### Generación de texto 📨

<video src="https://lena-voita.github.io/resources/lectures/lang_models/general/generation_example.mp4" controls loop>

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

In [ ]:
def get_likely_words(
    model: NgramModel, context: str, top_count: int = 10
) -> list[tuple]:
    """Dado un contexto obtiene las palabras más probables

    Params
    ------
    model: NgramModel
        Modelo con sus probabilidades calculadas
    context: str
        Contexto con el cual calcular las palabras más probables siguientes
    top_count: int
        Cantidad de palabras más probables. Default 10
    """
    history = tuple(context.split())
    predictions = dict(model[history]).items()
    return sorted(predictions, key=lambda prob: -1 * prob[1])[:top_count]

In [ ]:
get_likely_words(trigram_probs, "<s> the", top_count=1)

In [ ]:
# Strategy here
def get_next_word(words: list) -> str:
    return words[0][0]

In [ ]:
get_next_word(get_likely_words(trigram_probs, "emma was", 50))

In [ ]:
sentence = "<s> the"
for i in range(10):
    print(i, get_next_word(get_likely_words(trigram_probs, sentence, 50)))

#### Ejercicio 🤺: Genera lenguaje


- Utilizando el modelo de trigramas, diseña una estrategia para generación del lenguaje
- Implementa la fución `generate_text()` que reciba un modelo de n-gramas y una historia y genere texto utilizando la estrategia implementada.
- Agrega los parámetros que consideres necesarios a tu función de generación de texto

**Ejemplo:**

```python
sentence = "god was"
generate_text(trigram_probs, sentence)
>> god was evil 🐲
```

In [ ]:
from random import randint, uniform, choices

def generate_text(model: NgramModel, history: str):
    pass

In [ ]:
sentence = "<s> the"
print(sentence, end=" ")
generate_text(trigram_probs, sentence)

### Calculando la probabilidad de una oración

In [ ]:
def calculate_sentence_prob(sentence: list[str], model: NgramModel) -> float:
    prob = 0
    for w1, w2, w3 in ngrams(sentence, n=3):
        try:
            prob += np.log(model[w1, w2][w3])
        except KeyError:
            # OOV
            prob += 0.0
    return prob

In [ ]:
nltk.download("reuters")

In [ ]:
from nltk.corpus import reuters

In [ ]:
news_sentence = preprocess_sent(reuters.sents()[6701])
gutenberg_sentence = preprocess_sent(gutenberg.sents()[100])
sentences = [news_sentence, gutenberg_sentence, preprocess_sent(gutenberg.sents()[101])]

for sent in sentences:
    print(f"PROB={calculate_sentence_prob(sent, trigram_smooth)}: '{' '.join(sent)}'")

In [ ]:
i = 0

for j, sent in enumerate(reuters.sents()):
    sent = preprocess_sent(sent)
    if calculate_sentence_prob(sent, trigram_smooth) != -np.inf:
        print(
            f"{j} PROB={calculate_sentence_prob(sent, trigram_smooth)}: '{' '.join(sent)}'"
        )
        i += 1
    if i > 30:
        break

### Usando un dataset grandesito

In [2]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus-100", "en-es", split="train")

In [ ]:
text_prompt = "el gobierno"
# Procesamos la entrada del usuario
tokens = text_prompt.lower().split()
print(text_prompt, end=" ")
    
# Extraemos las dos últimas palabras del prompt y las codificamos
idxs = encode_sentence(tokens, word2idx)

w1 = idxs[-2]
w2 = idxs[-1]
context_idx = (w1, w2)

generate_text_int(
    model=int_smooth_trigram_model,
    context_idx=context_idx,
    idx2word=idx2word,
    strategy=get_next_random_word,
    max_tokens=10
)

el gobierno federal , provincial o territorial . </s> 


## Referencias

- [Maravilloso curso de Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html) ⚡